# 04 — Policy: decisões Cedar no Gateway

Referência: [AgentCore workshops](https://github.com/awslabs/agentcore-samples/tree/main/06-workshops). O código e o IaC desta jornada adaptam os conceitos às contas do workshop.

## Pré-requisito: 1 conta AWS

Esta aula não exige fronteiras entre contas. Seus stacks, integrações e permissões precisam estar implantados na conta da plataforma. Ela não funciona automaticamente em uma conta AWS vazia.

**Runner CodeBuild do projeto:** a implantação atual exige quatro contas e as integrações Registry, Terraform e RAG, inclusive com `--labs`. A seleção parcial não converte esse runner em uma implantação de conta única. Confira a [matriz de contas e execução](../docs/contas-e-elegibilidade.md).

![Arquitetura AWS da aula 04](../docs/diagramas/svg/04_policy_cedar.svg)

Leia as fronteiras do diagrama e confronte-as com o IaC e as chamadas abaixo.

Todos os serviços e agentes desta jornada são implantados na AWS.
O notebook é o roteiro e cliente das APIs; ferramentas, inferência,
políticas, fontes e memória executam nos serviços indicados.
Use o ambiente AWS descrito no [guia de implantação](../docs/deploy.md).
Configure os identificadores dos stacks antes de executar. Uma
dependência ausente bloqueia a aula antes das chamadas do exercício.
Antes de usar “Run All”, selecione as aulas compatíveis com as contas
preparadas. O bloco 10–14 exige quatro contas na implantação atual.

### Carregar a configuração e conferir as dependências

A configuração contém perfis/roles e IDs, nunca senhas ou tokens. O STS confirma a conta antes de consultar os stacks; o manifesto exige as dependências desta aula.

In [ ]:
import json
import os
import uuid
from pathlib import Path
from IPython.display import Code, display
import inspect
from santander_demo.aws_workshop.client import AWSWorkshop
from santander_demo.aws_workshop.config import load
from santander_demo.aws_workshop.manifest import require

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
CONFIG = Path(os.environ.get("WORKSHOP_CONFIG", ROOT / "config/aws.workshop.json"))
config = load(CONFIG)
dependencies = require(config, "04")
print({"lab": "04", "dependencies": list(dependencies), "region": config["region"]})

### Conectar ao ambiente implantado

Lemos os outputs exatos do CloudFormation. O cliente não cria infraestrutura e não substitui um serviço indisponível.

In [ ]:
lab = AWSWorkshop(CONFIG, domain="platform")
print(lab.preflight())

### Ler o Cedar que o IaC publica

As regras vinculam o principal ao sub, o cliente à audiência, a ação ao nome qualificado e o recurso ao ARN do Gateway. Observe o limite de entrada e a ausência de permit para D.

In [ ]:
from santander_demo.aws_workshop.contracts import policies
cedar = policies(lab.gateway["GatewayArn"], lab.identity["Issuer"],
                 lab.identity["SupervisorClientId"], lab.identity["RiscosClientId"],
                 {x: lab.identity[f"User{x.upper()}Oid"] for x in "abc"})
for name, text in cedar.items():
    print(name)
    display(Code(text, language="text"))

### Executar a matriz no Gateway gerenciado

Cada linha faz uma chamada HTTPS real. Validamos separadamente permit e uma negativa de Policy; infraestrutura indisponível interrompe o exercício.

In [ ]:
import time
since_ms = int(time.time() * 1000)
cases = [
    ("a", "supervisor", "riscos___consultar_risco", 1, True),
    ("b", "supervisor", "riscos___consultar_risco", 1, False),
    ("c", "supervisor", "especialista___consultar_fraude", 1, True),
    ("a", "supervisor", "especialista___consultar_fraude", 1, False),
    ("c", "riscos", "especialista___consultar_fraude", 1, False),
    ("d", "supervisor", "riscos___consultar_risco", 1, False),
    ("c", "supervisor", "riscos___consultar_risco", 11, False),
]
rows = []
for user, application, tool, limit, allowed in cases:
    jwt_token = lab.login(user, application)
    arguments = {"cliente_id": "C001", "max_registros": limit}
    evidence = (await lab.call(jwt_token, tool, arguments) if allowed else
                await lab.expect_denial(jwt_token, tool, arguments))
    rows.append({"user": user, "application": application, "tool": tool,
                 "expected": "ALLOW" if allowed else "DENY", "evidence": evidence})
print(json.dumps(rows, indent=2))

### Correlacionar Policy e execução da ferramenta

O ID AWS identifica a decisão nativa; request_id identifica o registro backend_invoked. Enviamos controles permitidos após as negativas e aguardamos sua entrega no CloudWatch. Ausência de logs sozinha não comprova uma negativa: exigimos também DENY nativo, controles entregues e uma janela adicional de observação.

In [ ]:
import asyncio
from santander_demo.aws_workshop.evidence import correlate_policy
for tool in ["riscos___consultar_risco", "especialista___consultar_fraude"]:
    rows.append({"user": "c", "application": "supervisor", "tool": tool,
                 "expected": "ALLOW", "control_after_denials": True,
                 "evidence": await lab.call(lab.login("c"), tool,
                     {"cliente_id": "C001", "max_registros": 1})})
stable_since = None
for attempt in range(48):
    correlate_policy(lab, rows, since_ms)
    ready = all(row["correlation"]["native_policy_verified"]
                and row["correlation"]["backend_observed"] == (row["expected"] == "ALLOW")
                for row in rows)
    stable_since = (stable_since or time.monotonic()) if ready else None
    if stable_since is not None and time.monotonic() - stable_since >= 20:
        break
    await asyncio.sleep(5)
else:
    raise AssertionError("CORRELACAO_POLICY_BACKEND_INCOMPLETA")
print(json.dumps({"observation_after_controls_seconds": 20, "rows": rows}, indent=2))

**Exercício:** explique qual condição nega cada linha. Para mudar uma regra, edite o gerador IaC, revise o diff e faça o deploy do stack `bindings`; não edite a Policy diretamente pela API.

### Evidências e encerramento

Registre IDs de requisição e decisões, sem tokens ou senhas. Os recursos são compartilhados entre aulas: a remoção pertence ao pipeline de limpeza do projeto e deve respeitar essa dependência.